In [ ]:
# Databricks notebook source
# %python

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import re

# ---------------------------
# CONFIG
# ---------------------------
RAW_BASE = "/Volumes/msbabigdata/spark/trends"
DELTA_BASE = "/Volumes/msbabigdata/spark/trends/delta"

# Set these for your environment
CATALOG = "main"          # e.g. "main" or your UC catalog
SCHEMA = "faers_dashboard" # e.g. "faers_dashboard"

# ---------------------------
# SETUP
# ---------------------------
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

def fqtn(table_name: str) -> str:
    return f"{CATALOG}.{SCHEMA}.{table_name}"

def normalize_cols(df: DataFrame) -> DataFrame:
    return df.toDF(*[c.strip().lower() for c in df.columns])

def read_raw(name: str) -> DataFrame:
    """
    Reads parquet from RAW_BASE with common case variants:
    demo.parquet, Demo.parquet, DEMO.parquet, etc.
    """
    candidates = []
    base_names = [name, name.lower(), name.upper(), name.capitalize(), name.title()]
    for bn in base_names:
        candidates.append(f"{RAW_BASE}/{bn}.parquet")
        candidates.append(f"{RAW_BASE}/{bn}")
    last_err = None
    for p in candidates:
        try:
            df = spark.read.parquet(p)
            print(f"Loaded {name} from {p} ({df.count():,} rows)")
            return normalize_cols(df)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not read raw parquet for '{name}' from {RAW_BASE}. Last error: {last_err}")

def year_q_expr(df: DataFrame):
    cols = set(df.columns)
    if "year_q" in cols:
        return F.upper(F.regexp_replace(F.trim(F.col("year_q").cast("string")), " ", ""))
    if "quarter" in cols:
        return F.upper(F.regexp_replace(F.trim(F.col("quarter").cast("string")), " ", ""))
    return F.lit("")

def safe_col(df: DataFrame, col_name: str, cast_type: str = "string"):
    if col_name in df.columns:
        return F.col(col_name).cast(cast_type)
    return F.lit("").cast(cast_type)

# Canonical manufacturer normalization (Spark SQL version of your Python logic)
SUFFIXES = [
    "inc", "incorporated", "corp", "corporation", "company", "co", "ltd", "limited",
    "llc", "plc", "ag", "gmbh", "sa", "nv", "bv", "oyj", "kk"
]
suffix_regex = "(" + "|".join([re.escape(s) for s in SUFFIXES]) + ")"
# remove one or more trailing suffix tokens
trail_suffix_pattern = rf"(\s+{suffix_regex})+$"

def canonicalize_mfr(col):
    cleaned = F.lower(F.regexp_replace(F.coalesce(col.cast("string"), F.lit("")), r"[^a-z0-9\s]", " "))
    collapsed = F.trim(F.regexp_replace(cleaned, r"\s+", " "))
    no_suffix = F.trim(F.regexp_replace(collapsed, trail_suffix_pattern, ""))
    return no_suffix

def write_delta_and_register(df: DataFrame, table_name: str):
    path = f"{DELTA_BASE}/{table_name}"
    target = fqtn(table_name)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path)
    )

    spark.sql(f"DROP TABLE IF EXISTS {target}")
    spark.sql(f"CREATE TABLE {target} USING DELTA LOCATION '{path}'")
    n = spark.table(target).count()
    print(f"Wrote {target} at {path} ({n:,} rows)")

def listify_lookup(df: DataFrame, key_col: str) -> DataFrame:
    # columns expected: key_col, primaryid
    return (
        df.groupBy(key_col)
          .agg(F.sort_array(F.collect_set(F.col("primaryid").cast("string"))).alias("primaryids"))
          .withColumn("n_cases", F.size(F.col("primaryids")))
    )

# ---------------------------
# READ RAW SOURCES
# ---------------------------
demo = read_raw("demo")
drug = read_raw("drug")
reac = read_raw("reac")
outc = read_raw("outc")
rpsr = read_raw("rpsr")
indi = read_raw("indi")

# ---------------------------
# BUILD SLIM TABLES
# ---------------------------
demo_slim = (
    demo.select(
        safe_col(demo, "primaryid").alias("primaryid"),
        year_q_expr(demo).alias("year_q"),
        safe_col(demo, "event_dt").alias("event_dt"),
        safe_col(demo, "sex").alias("sex"),
        safe_col(demo, "age", "double").alias("age"),
        safe_col(demo, "occr_country").alias("occr_country"),
        safe_col(demo, "mfr_sndr").alias("mfr_sndr"),
        safe_col(demo, "lit_ref").alias("lit_ref"),
    )
    .withColumn("canonical_mfr", canonicalize_mfr(F.col("mfr_sndr")))
    .dropDuplicates()
)

drug_records_slim = (
    drug.select(
        safe_col(drug, "primaryid").alias("primaryid"),
        year_q_expr(drug).alias("year_q"),
        F.upper(safe_col(drug, "role_cod")).alias("role_cod"),
        safe_col(drug, "drugname").alias("drugname"),
        F.lower(F.trim(safe_col(drug, "drugname"))).alias("drugname_norm"),
        safe_col(drug, "prod_ai").alias("prod_ai"),
        F.lower(F.trim(safe_col(drug, "prod_ai"))).alias("prod_ai_norm"),
        safe_col(drug, "route").alias("route"),
        safe_col(drug, "dose_amt").alias("dose_amt"),
        safe_col(drug, "dose_unit").alias("dose_unit"),
        safe_col(drug, "dose_form").alias("dose_form"),
        safe_col(drug, "dose_freq").alias("dose_freq"),
        safe_col(drug, "mfr_sndr").alias("mfr_sndr"),
    )
    .withColumn("canonical_mfr", canonicalize_mfr(F.col("mfr_sndr")))
    .dropDuplicates()
)

reac_slim = (
    reac.select(
        safe_col(reac, "primaryid").alias("primaryid"),
        year_q_expr(reac).alias("year_q"),
        safe_col(reac, "pt").alias("pt"),
        F.lower(F.trim(safe_col(reac, "pt"))).alias("pt_norm"),
    )
    .dropDuplicates()
)

outc_slim = (
    outc.select(
        safe_col(outc, "primaryid").alias("primaryid"),
        year_q_expr(outc).alias("year_q"),
        F.upper(safe_col(outc, "outc_cod")).alias("outc_cod"),
    )
    .dropDuplicates()
)

rpsr_slim = (
    rpsr.select(
        safe_col(rpsr, "primaryid").alias("primaryid"),
        year_q_expr(rpsr).alias("year_q"),
        F.upper(safe_col(rpsr, "rpsr_cod")).alias("rpsr_cod"),
    )
    .dropDuplicates()
)

indi_slim = (
    indi.select(
        safe_col(indi, "primaryid").alias("primaryid"),
        year_q_expr(indi).alias("year_q"),
        safe_col(indi, "indi_pt").alias("indi_pt"),
        F.lower(F.trim(safe_col(indi, "indi_pt"))).alias("indi_pt_norm"),
    )
    .dropDuplicates()
)

# ---------------------------
# PRECOMPUTE TABLES
# ---------------------------
drug_summary = (
    drug_records_slim.groupBy("drugname")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
    .orderBy(F.desc("n_cases"))
)

reac_summary = (
    reac_slim.groupBy("pt")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
    .orderBy(F.desc("n_cases"))
)

manufacturer_summary = (
    demo_slim.where(F.trim(F.col("canonical_mfr")) != "")
    .groupBy("canonical_mfr")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
    .orderBy(F.desc("n_cases"))
)

fact_drug_quarter = (
    drug_records_slim.groupBy("drugname", "year_q")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
)

fact_reac_quarter = (
    reac_slim.groupBy("pt", "year_q")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
)

fact_manufacturer_quarter = (
    demo_slim.where(F.trim(F.col("canonical_mfr")) != "")
    .groupBy("canonical_mfr", "year_q")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
)

lookup_quarter_cases = listify_lookup(
    demo_slim.select("year_q", "primaryid").dropDuplicates(), "year_q"
)

lookup_drug_cases = listify_lookup(
    drug_records_slim.select("drugname", "primaryid").dropDuplicates(), "drugname"
)

lookup_drug_role_cases = listify_lookup(
    drug_records_slim.select(
        F.concat_ws("|", F.col("drugname"), F.col("role_cod")).alias("drug_role_key"),
        F.col("primaryid"),
    ).dropDuplicates(),
    "drug_role_key",
)

lookup_reaction_cases = listify_lookup(
    reac_slim.select("pt", "primaryid").dropDuplicates(), "pt"
)

lookup_manufacturer_cases = listify_lookup(
    demo_slim.where(F.trim(F.col("canonical_mfr")) != "")
    .select("canonical_mfr", "primaryid")
    .dropDuplicates(),
    "canonical_mfr",
)

manufacturer_name_lookup = (
    demo_slim.where(F.trim(F.col("mfr_sndr")) != "")
    .groupBy("mfr_sndr", "canonical_mfr")
    .agg(F.countDistinct("primaryid").alias("n_cases"))
    .orderBy(F.desc("n_cases"))
)

dose_bucket_slim = (
    drug_records_slim.select("primaryid", "dose_amt", "dose_unit")
    .withColumn(
        "dose_amt_bucket",
        F.when(F.trim(F.col("dose_amt")) == "", F.lit("Not reported")).otherwise(F.trim(F.col("dose_amt")))
    )
    .withColumn("dose_unit_norm", F.lower(F.trim(F.col("dose_unit"))))
)

global_kpis = (
    demo_slim.agg(
        F.countDistinct("primaryid").alias("cases"),
        F.min("year_q").alias("quarter_min"),
        F.max("year_q").alias("quarter_max"),
    )
    .crossJoin(
        outc_slim.where(F.col("outc_cod") == "DE").agg(
            F.countDistinct("primaryid").alias("deaths")
        )
    )
    .crossJoin(
        drug_records_slim.agg(F.countDistinct("drugname").alias("unique_drugs"))
    )
    .crossJoin(
        reac_slim.agg(F.countDistinct("pt").alias("unique_reactions"))
    )
)

drug_name_lookup = (
    drug_records_slim
    .select("drugname", "drugname_norm", "prod_ai", "prod_ai_norm")
    .dropDuplicates()
)

# ---------------------------
# WRITE ALL DELTA TABLES
# ---------------------------
outputs = {
    "demo_slim": demo_slim,
    "drug_records_slim": drug_records_slim,
    "reac_slim": reac_slim,
    "outc_slim": outc_slim,
    "rpsr_slim": rpsr_slim,
    "indi_slim": indi_slim,
    "drug_summary": drug_summary,
    "reac_summary": reac_summary,
    "manufacturer_summary": manufacturer_summary,
    "fact_drug_quarter": fact_drug_quarter,
    "fact_reac_quarter": fact_reac_quarter,
    "fact_manufacturer_quarter": fact_manufacturer_quarter,
    "lookup_quarter_cases": lookup_quarter_cases,
    "lookup_drug_cases": lookup_drug_cases,
    "lookup_drug_role_cases": lookup_drug_role_cases,
    "lookup_reaction_cases": lookup_reaction_cases,
    "lookup_manufacturer_cases": lookup_manufacturer_cases,
    "manufacturer_name_lookup": manufacturer_name_lookup,
    "dose_bucket_slim": dose_bucket_slim,
    "global_kpis": global_kpis,
    "drug_name_lookup": drug_name_lookup,
}

for name, frame in outputs.items():
    write_delta_and_register(frame, name)

# ---------------------------
# OPTIMIZE (major/high-use tables)
# ---------------------------
optimize_tables = [
    "demo_slim",
    "drug_records_slim",
    "reac_slim",
    "outc_slim",
    "rpsr_slim",
    "indi_slim",
    "fact_drug_quarter",
    "fact_reac_quarter",
    "drug_name_lookup",
    "manufacturer_name_lookup",
]

for t in optimize_tables:
    spark.sql(f"OPTIMIZE {fqtn(t)}")
    print(f"OPTIMIZE complete: {fqtn(t)}")

print("\nBootstrap complete.")
print(f"Catalog.Schema: {CATALOG}.{SCHEMA}")
print(f"Delta base path: {DELTA_BASE}")